# 🗳️ Ingestão Integral de Logs de Urna UF a UF — Google Colab (100% das Urnas)

Este notebook é **100% autocontido** e projetado para rodar no **Google Colab** (aproveitando os 200 GB de disco efêmero ou Google Drive), processando **100% dos logs de todas as urnas** de cada Unidade Federativa (UF) sem amostragem (`MAX_JEZ_FILES = None`), e sincronizando automaticamente as novas partições no dataset do Kaggle utilizando a **Kaggle Python SDK** (`dir_mode='tar'`).

> ⚡ **100% Python Nativo:** Não utiliza comandos shell (`!kaggle`). O ciclo de vida de autenticação, download, parsing, redação LGPD e upload é orquestrado inteiramente via Python SDK.

---

## 🔑 Passo 1: Configurar Token do Kaggle nos Secrets do Colab

1. No menu lateral esquerdo do Google Colab, clique no ícone de chave (**🔑 Secrets**).
2. Adicione os dois segredos abaixo:
   - **`KAGGLE_USERNAME`**: seu nome de usuário no Kaggle (ex.: `daniellmattos`)
   - **`KAGGLE_KEY`**: sua API Key do Kaggle (gerada em [kaggle.com/settings](https://www.kaggle.com/settings) -> *Create New Token*)
3. Ative a chave seletora **"Notebook access"** para ambos os segredos.

> 💡 **Dica:** Se desejar persistir o acervo entre desconexões do Colab, ative a opção `USE_GOOGLE_DRIVE = True` no formulário abaixo.

In [ ]:
# 1. Instalação das bibliotecas necessárias no ambiente Colab
!pip install -q polars pyarrow httpx py7zr kaggle tqdm

import os
from pathlib import Path
from kaggle.api.kaggle_api_extended import KaggleApi

# 2. Carrega credenciais dos Secrets do Google Colab para as variáveis de ambiente
kaggle_user = None
kaggle_key = None

try:
    from google.colab import userdata
    try:
        kaggle_user = userdata.get('KAGGLE_USERNAME')
        kaggle_key = userdata.get('KAGGLE_KEY')
    except Exception:
        pass
except ImportError:
    pass

kaggle_user = kaggle_user or os.environ.get('KAGGLE_USERNAME')
kaggle_key = kaggle_key or os.environ.get('KAGGLE_KEY')

if kaggle_user and kaggle_key:
    os.environ["KAGGLE_USERNAME"] = kaggle_user
    os.environ["KAGGLE_KEY"] = kaggle_key
    api = KaggleApi()
    api.authenticate()
    print(f"✅ Kaggle Python SDK autenticado com sucesso para: {kaggle_user}")
else:
    raise ValueError(
        "❌ KAGGLE_USERNAME e KAGGLE_KEY não encontrados! "
        "Adicione os secrets no menu lateral esquerdo (🔑 Secrets) do Google Colab."
    )

## ⚙️ Passo 2: Definir as UFs e Parâmetros de Execução

Configure a UF (ou lista de UFs separadas por vírgula) que deseja processar. Para processar 100% das urnas sem qualquer amostragem, mantenha `MAX_JEZ_FILES = None`.

In [ ]:
#@title Parâmetros de Ingestão por UF (Colab Form)
UF_INPUT = "AC" #@param {type:"string"}
# Dica: Você pode informar uma UF única ("AC") ou múltiplas ("AC, AP, RR") ou "ALL"

ALL_UFS = [
    "AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO", "MA",
    "MG", "MS", "MT", "PA", "PB", "PE", "PI", "PR", "RJ", "RN",
    "RO", "RR", "RS", "SC", "SE", "SP", "TO", "ZZ"
]

if UF_INPUT.strip().upper() == "ALL":
    TARGET_UFS = ALL_UFS
else:
    TARGET_UFS = [u.strip().upper() for u in UF_INPUT.split(",") if u.strip()]

TARGET_TURNS = [1, 2] # Processa 1º e 2º turnos
ELECTION_YEAR = 2026
MAX_JEZ_FILES = None # None = 100% de todas as urnas (sem amostragem)
DATASET_SLUG = "daniellmattos/tse-analysis-eleicoes-2026" #@param {type:"string"}
USE_GOOGLE_DRIVE = False #@param {type:"boolean"}
SYNC_TO_KAGGLE = True #@param {type:"boolean"}
UPLOAD_AFTER_EACH_UF = False #@param {type:"boolean"}
# Se False, acumula todas as UFs selecionadas e faz 1 único upload ao final

# Define diretórios de trabalho
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_WORKSPACE = Path("/content/drive/MyDrive/tse_eleicoes_dataset")
else:
    BASE_WORKSPACE = Path("/content/tse_eleicoes_dataset")

STAGING_DIR = Path("/content/staging")
EXPORTS_DIR = BASE_WORKSPACE / "exports"
BASE_WORKSPACE.mkdir(parents=True, exist_ok=True)
STAGING_DIR.mkdir(parents=True, exist_ok=True)
EXPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Alvos configurados: UFs={TARGET_UFS}, Turnos={TARGET_TURNS}, Ano={ELECTION_YEAR}")
print(f"Diretório de acumulação local: {EXPORTS_DIR}")

In [ ]:
# 1. Baixa o dataset base atual do Kaggle se o diretório local ainda não contiver as tabelas
if not (EXPORTS_DIR / "fact_boletim_urna").exists():
    print(f"📥 Baixando acervo existente do dataset '{DATASET_SLUG}' via Kaggle SDK...")
    api.dataset_download_files(DATASET_SLUG, path=str(EXPORTS_DIR), unzip=True)
    print("✅ Acervo base baixado e descompactado em:", EXPORTS_DIR)
else:
    print("ℹ️ Acervo base já presente localmente em:", EXPORTS_DIR)

# 2. Garante a presença do arquivo dataset-metadata.json
meta_file = EXPORTS_DIR / "dataset-metadata.json"
if not meta_file.exists():
    print(f"📥 Baixando metadados oficiais do dataset '{DATASET_SLUG}'...")
    api.dataset_metadata(DATASET_SLUG, path=str(EXPORTS_DIR))
    print("✅ Metadados salvos em:", meta_file)
else:
    print("ℹ️ Metadados já existentes em:", meta_file)

## 🚀 Passo 3: Motor de Ingestão Autocontido (Streaming & Extração em Memória)

O módulo a seguir baixa o pacote oficial do TSE (`bu_imgbu_logjez_rdv_vscmr_*.zip`) diretamente para disco temporário, descompacta os arquivos `.logjez` em memória (com suporte a 7z/LZMA e ZIP), aplica a sanitização LGPD (`<ID_REDACTED>`), remove a coluna de hardware `mac_code` e grava partições Hive Parquet ZSTD de alta performance.

In [ ]:
import io
import re
import shutil
import time
import zipfile
from collections.abc import Sequence
from pathlib import Path

import httpx
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
import py7zr
from tqdm.auto import tqdm

SEVEN_ZIP_MAGIC: bytes = b"7z\xbc\xaf\x27\x1c"
ZIP_MAGIC: bytes = b"PK\x03\x04"

JEZ_ENTRY_RE = re.compile(
    r"^o(?P<pleito>\d{5})(?P<uf>[a-zA-Z]{2})(?P<mun>\d{5})(?P<zona>\d{4})(?P<secao>\d{4})-log\.jez$",
    re.IGNORECASE,
)
_ROTATED_LOG_RE = re.compile(r"^logd\.dat\.(\d+)(?:\.(?:jez|zip|7z))?$", re.IGNORECASE)
_CURRENT_LOG_RE = re.compile(r"^logd\.dat$", re.IGNORECASE)

FACT_URNA_LOGS_SCHEMA = {
    "election_year": pl.Int32,
    "turn": pl.Int8,
    "uf": pl.Utf8,
    "cd_pleito": pl.Int64,
    "cd_municipio": pl.Int32,
    "nr_zona": pl.Int32,
    "nr_secao": pl.Int32,
    "source_jez_file": pl.Utf8,
    "source_log_member": pl.Utf8,
    "file_seq": pl.Int64,
    "raw_timestamp": pl.Utf8,
    "log_level": pl.Utf8,
    "urna_serial": pl.Utf8,
    "application": pl.Utf8,
    "message_text": pl.Utf8,
}

FACT_URNA_LOGS_ARROW_SCHEMA = pa.schema([
    ("election_year", pa.int32()),
    ("turn", pa.int8()),
    ("uf", pa.string()),
    ("cd_pleito", pa.int64()),
    ("cd_municipio", pa.int32()),
    ("nr_zona", pa.int32()),
    ("nr_secao", pa.int32()),
    ("source_jez_file", pa.string()),
    ("source_log_member", pa.string()),
    ("file_seq", pa.int64()),
    ("raw_timestamp", pa.string()),
    ("log_level", pa.string()),
    ("urna_serial", pa.string()),
    ("application", pa.string()),
    ("message_text", pa.string()),
])

def sanitize_message_text(message: str) -> str:
    """Mascaramento estrito de IDs numéricos de 5+ dígitos para conformidade LGPD."""
    return re.sub(r"\b\d{5,}\b", "<ID_REDACTED>", message)

def sort_rotated_log_names(names: Sequence[str]) -> list[str]:
    """Ordena deterministicamente os segmentos de log: logd.dat.2 -> logd.dat.1 -> logd.dat."""
    rotated = []
    current = []
    for name in names:
        base = Path(name).name
        m_rot = _ROTATED_LOG_RE.match(base)
        if m_rot:
            rotated.append((int(m_rot.group(1)), name))
        elif _CURRENT_LOG_RE.match(base):
            current.append(name)
    rotated.sort(key=lambda item: item[0], reverse=True)
    return [name for _, name in rotated] + sorted(current)

def extract_log_members_from_jez_bytes(jez_bytes: bytes) -> list[tuple[str, bytes]]:
    """Extrai membros de log em memória inspecionando magic bytes (7z ou ZIP)."""
    members: list[tuple[str, bytes]] = []
    if jez_bytes.startswith(SEVEN_ZIP_MAGIC):
        with py7zr.SevenZipFile(io.BytesIO(jez_bytes), mode="r") as archive:
            extracted = archive.readall()
            for name in sort_rotated_log_names(list(extracted.keys())):
                stream = extracted[name]
                members.append((Path(name).name, stream.read()))
    elif jez_bytes.startswith(ZIP_MAGIC):
        with zipfile.ZipFile(io.BytesIO(jez_bytes), mode="r") as zf:
            for name in sort_rotated_log_names(zf.namelist()):
                members.append((Path(name).name, zf.read(name)))
    return members

def parse_logd_lines(raw_bytes: bytes, base_row: dict, start_seq: int = 1) -> tuple[list[dict], int]:
    """Decodifica linhas tabuladas de logd.dat aplicando redação de PII e omitindo mac_code."""
    rows = []
    seq = start_seq
    text = raw_bytes.decode("utf-8", errors="replace")
    for line in text.splitlines():
        parts = line.split("\t")
        if len(parts) >= 5:
            raw_ts = parts[0].strip()
            level = parts[1].strip()
            serial = parts[2].strip()
            app = parts[3].strip()
            msg = sanitize_message_text(parts[4].strip())
            row = dict(base_row)
            row["file_seq"] = seq
            row["raw_timestamp"] = raw_ts
            row["log_level"] = level
            row["urna_serial"] = serial
            row["application"] = app
            row["message_text"] = msg
            rows.append(row)
            seq += 1
    return rows, seq - start_seq

print("✅ Módulos de extração, sanitização LGPD e parsing configurados com sucesso.")

In [ ]:
url_template = "https://cdn.tse.jus.br/estatistica/sead/eleicoes/eleicoes{election_year}/arqurnatot/bu_imgbu_logjez_rdv_vscmr_{election_year}_{turn}t_{uf}.zip"

def upload_dataset_version(notes: str):
    """Envia uma nova versão do dataset ao Kaggle usando o SDK Python."""
    print(f"\n📤 [Kaggle SDK] Publicando nova versão no dataset '{DATASET_SLUG}'...")
    res = api.dataset_create_version(
        folder=str(EXPORTS_DIR),
        version_notes=notes,
        dir_mode="tar",
        convert_to_csv=False
    )
    print(f"✅ [Kaggle SDK] Versão criada com sucesso! Resposta: {res}")

global_processed_urnas = 0
global_processed_lines = 0

for current_uf in TARGET_UFS:
    uf_urnas = 0
    uf_lines = 0

    print(f"\n{'#'*80}")
    print(f"INICIANDO PROCESSAMENTO DA UF: {current_uf}")
    print(f"{'#'*80}")

    for turn in TARGET_TURNS:
        zip_url = url_template.format(election_year=ELECTION_YEAR, turn=turn, uf=current_uf.lower())
        out_parquet = EXPORTS_DIR / "fact_urna_logs" / f"election_year={ELECTION_YEAR}" / f"turn={turn}" / f"uf={current_uf.upper()}" / "part-00000.parquet"
        out_parquet.parent.mkdir(parents=True, exist_ok=True)
        local_zip = STAGING_DIR / f"bundle_{ELECTION_YEAR}_{turn}t_{current_uf}.zip"

        print(f"\n--- UF={current_uf} | Turno={turn} ---")
        print(f"URL TSE: {zip_url}")

        # 1. Download do ZIP oficial da UF
        print(f"[1/4] Baixando arquivo da UF {current_uf}...")
        t0 = time.time()
        try:
            with httpx.stream("GET", zip_url, timeout=300.0, follow_redirects=True) as resp:
                if resp.status_code == 404:
                    print(f"ℹ️ Turno {turn} inexistente para {current_uf} no TSE (404). Pulando.")
                    continue
                resp.raise_for_status()
                with open(local_zip, "wb") as f:
                    for chunk in resp.iter_bytes(chunk_size=1024*1024):
                        f.write(chunk)
        except Exception as e:
            print(f"⚠️ Falha no download de {zip_url}: {e}")
            continue

        dl_sec = time.time() - t0
        zip_size_mb = local_zip.stat().st_size / (1024 * 1024)
        print(f"✅ Download concluído: {zip_size_mb:.2f} MB em {dl_sec:.1f}s ({zip_size_mb/max(1, dl_sec):.2f} MB/s)")

        # 2. Inspeciona e processa urnas
        print(f"[2/4] Extraindo e processando logs das urnas...")
        accumulated_rows = []
        dfs_chunks = []
        part_urnas = 0
        part_lines = 0

        with zipfile.ZipFile(local_zip, mode="r") as zf:
            jez_entries = [n for n in zf.namelist() if n.lower().endswith(".logjez") or n.lower().endswith(".jez")]
            if MAX_JEZ_FILES is not None:
                jez_entries = jez_entries[:MAX_JEZ_FILES]
            total_jez = len(jez_entries)
            print(f"Total de urnas a processar nesta partição: {total_jez:,}")

            t_proc = time.time()
            pbar = tqdm(total=total_jez, desc=f"{current_uf} T{turn}", unit="urna")
            for entry_name in jez_entries:
                m = JEZ_ENTRY_RE.match(entry_name)
                cd_pleito = int(m.group("pleito")) if m else 0
                cd_mun = int(m.group("mun")) if m else 0
                nr_zona = int(m.group("zona")) if m else 0
                nr_secao = int(m.group("secao")) if m else 0

                jez_bytes = zf.read(entry_name)
                members = extract_log_members_from_jez_bytes(jez_bytes)
                current_seq = 1
                for mem_name, raw_log in members:
                    base_row = {
                        "election_year": ELECTION_YEAR,
                        "turn": turn,
                        "uf": current_uf.upper(),
                        "cd_pleito": cd_pleito,
                        "cd_municipio": cd_mun,
                        "nr_zona": nr_zona,
                        "nr_secao": nr_secao,
                        "source_jez_file": entry_name,
                        "source_log_member": mem_name,
                    }
                    rows, count = parse_logd_lines(raw_log, base_row, start_seq=current_seq)
                    accumulated_rows.extend(rows)
                    current_seq += count
                    part_lines += count

                part_urnas += 1
                pbar.update(1)

                if len(accumulated_rows) >= 500_000:
                    dfs_chunks.append(pl.DataFrame(accumulated_rows, schema=FACT_URNA_LOGS_SCHEMA))
                    accumulated_rows = []

            pbar.close()

        if accumulated_rows:
            dfs_chunks.append(pl.DataFrame(accumulated_rows, schema=FACT_URNA_LOGS_SCHEMA))
            accumulated_rows = []

        proc_sec = time.time() - t_proc
        print(f"✅ Processamento: {part_urnas:,} urnas ({part_lines:,} linhas) em {proc_sec:.1f}s")

        # 3. Escrita atômica em Parquet ZSTD nível 5
        print(f"[3/4] Gravando Parquet ZSTD em: {out_parquet}")
        tmp_pq = out_parquet.with_suffix(".parquet.tmp")
        with pq.ParquetWriter(tmp_pq, schema=FACT_URNA_LOGS_ARROW_SCHEMA, compression="zstd", compression_level=5, use_dictionary=True) as writer:
            for chunk in dfs_chunks:
                writer.write_table(chunk.to_arrow().cast(FACT_URNA_LOGS_ARROW_SCHEMA))
        tmp_pq.replace(out_parquet)
        dfs_chunks.clear()

        pq_size_mb = out_parquet.stat().st_size / (1024 * 1024)
        print(f"✅ Parquet gravado com sucesso: {pq_size_mb:.2f} MB | {part_lines:,} linhas")

        # 4. Limpeza imediata do arquivo ZIP bruto para liberar disco
        print(f"[4/4] Limpando arquivo ZIP temporário para preservar espaço em disco...")
        local_zip.unlink(missing_ok=True)

        uf_urnas += part_urnas
        uf_lines += part_lines

    global_processed_urnas += uf_urnas
    global_processed_lines += uf_lines
    print(f"🎉 UF {current_uf} concluída: {uf_urnas:,} urnas, {uf_lines:,} linhas.")

    # Upload após cada UF (se configurado)
    if SYNC_TO_KAGGLE and UPLOAD_AFTER_EACH_UF:
        note = f"Append 100% logs da UF {current_uf} ({uf_urnas:,} urnas, {uf_lines:,} linhas)"
        upload_dataset_version(note)

print(f"\n{'='*80}")
print(f"PROCESSAMENTO COMPLETO DE TODAS AS UFS SELECIONADAS!")
print(f"Total Urnas: {global_processed_urnas:,} | Total Linhas: {global_processed_lines:,}")
print(f"{'='*80}")

# Upload consolidado do lote (se UPLOAD_AFTER_EACH_UF for False)
if SYNC_TO_KAGGLE and not UPLOAD_AFTER_EACH_UF and global_processed_urnas > 0:
    note = f"Append lote UFs {', '.join(TARGET_UFS)} ({global_processed_urnas:,} urnas, {global_processed_lines:,} linhas)"
    upload_dataset_version(note)

## 🔍 Passo 4: Validação do Acervo e Próximos Passos

Após a publicação no Kaggle, o acervo de logs brutos estará completo e disponível para reexecução dos notebooks Silver e Gold (`NB 05` a `NB 09`).

In [ ]:
logs_root = EXPORTS_DIR / "fact_urna_logs"
if logs_root.exists():
    parquets = sorted(logs_root.glob("**/*.parquet"))
    total_bytes = sum(p.stat().st_size for p in parquets)
    print(f"📊 Total de arquivos Parquet em fact_urna_logs: {len(parquets)}")
    print(f"💾 Espaço total ocupado: {total_bytes / (1024**3):.2f} GB")
    print("\nÚltimas partições geradas:")
    for p in parquets[-10:]:
        rel = p.relative_to(EXPORTS_DIR)
        size_mb = p.stat().st_size / (1024 * 1024)
        print(f"  - {rel} ({size_mb:.2f} MB)")
else:
    print("Nenhuma partição encontrada em fact_urna_logs.")